In [0]:
# ============================================================================
# NOTEBOOK: VISUALISATION COUCHE GOLD - Qualité de l'Eau
# Objectif: Générer une visualisation interactives à partir des tables Delta
#          créées par le notebook d'ingestion précédent
# ============================================================================

# MAGIC %md
# MAGIC ## Dashboard Qualité de l'Eau - Couche Gold
# MAGIC 
# MAGIC Ce notebook lit les tables Delta managées créées à l'étape précédente et génère une visualisation Plotly.

# ============================================================================
# 1. DEPENDANCES ET CONFIGURATION
# ============================================================================

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Noms exacts des tables Delta créées par le script d'ingestion
GOLD_TABLES = {
    "kpis": "gold_kpis",
    "departements": "gold_departements", 
    "communes_best": "gold_communes_best",
    "communes_worst": "gold_communes_worst",
    "critical": "gold_critical"
}

# Fonction utilitaire pour trouver une colonne malgré les variations de nom
def safe_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# ============================================================================
# 2. CHARGEMENT DES TABLES DELTA
# ============================================================================

print("Chargement des tables Delta Gold...")
tables = {}

for key, table_name in GOLD_TABLES.items():
    try:
        # Lecture directe depuis le catalogue Delta
        df_spark = spark.table(table_name)
        df_pandas = df_spark.toPandas()
        
        # Nettoyage des colonnes de métadonnées ajoutées par l'ingestion
        meta_cols = ['loaded_at', 'source_file']
        df_pandas.drop(columns=[c for c in meta_cols if c in df_pandas.columns], inplace=True, errors='ignore')
        
        tables[key] = df_pandas
        print(f"{table_name}: {len(df_pandas)} lignes, {len(df_pandas.columns)} colonnes")
    except Exception as e:
        print(f"Erreur sur {table_name}: {e}")
        tables[key] = None

# ============================================================================
# 3. KPI PRINCIPAL - Taux de Conformité Global
# ============================================================================

# MAGIC %md
# MAGIC ### Indicateurs Clés

if tables.get("kpis") is not None and not tables["kpis"].empty:
    kpi = tables["kpis"].iloc[0]
    
    # Récupération sécurisée des colonnes
    conformity = kpi.get('global_conformity_rate', 0)
    total_samples = kpi.get('total_samplings', kpi.get('total_records', 0))
    depts = kpi.get('departments_covered', 'N/A')
    communes = kpi.get('communes_covered', 'N/A')
    alerts = kpi.get('quality_alert', kpi.get('alerts', 0))
    
    # Jauge de conformité
    fig = go.Figure()
    fig.add_trace(go.Indicator(
        mode="gauge+number+delta",
        value=conformity,
        domain={'x': [0, 1], 'y': [0, 1]},
        title={'text': "Taux de Conformité Global (%)"},
        delta={'reference': 90, 'increasing': {'color': "RebeccaPurple"}},
        gauge={
            'axis': {'range': [None, 100]},
            'bar': {'color': "darkgreen" if conformity >= 90 else "orange" if conformity >= 70 else "red"},
            'steps': [
                {'range': [0, 70], 'color': "lightgray"},
                {'range': [70, 90], 'color': "gray"}],
            'threshold': {
                'line': {'color': "red", 'width': 4},
                'thickness': 0.75,
                'value': 90}}))
    
    fig.update_layout(height=300, margin=dict(l=20, r=30, t=50, b=20))
    fig.show()
    
    print(f"\nResume des KPIs :")
    print(f"   - Prelevements analyses : {total_samples:,}")
    print(f"   - Departements couverts : {depts}")
    print(f"   - Communes couvertes : {communes}")
    print(f"   - Alertes qualite : {alerts}")



Chargement des tables Delta Gold...
gold_kpis: 1 lignes, 13 colonnes
gold_departements: 103 lignes, 9 colonnes
gold_communes_best: 10 lignes, 3 colonnes
gold_communes_worst: 10 lignes, 3 colonnes
gold_critical: 1 lignes, 10 colonnes



Resume des KPIs :
   - Prelevements analyses : 295,089
   - Departements couverts : 103
   - Communes couvertes : 30152
   - Alertes qualite : 0
